# Compare the three churn models

Run `02a_logistic_regression.ipynb`, `02b_random_forest.ipynb` and `02c_gradient_boosted_trees.ipynb` first.
Each one writes a JSON file under `data/processed/model_results/`. This notebook only reads those files.
It does not train again, and it does not choose a winner by the test score. Each model already chose its own settings by cross-validated AUC.

Name:

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not find the project folder.")


sns.set_theme(style="whitegrid")
result_dir = find_project_root() / "data" / "processed" / "model_results"
expected = [
    "logistic_regression.json",
    "random_forest.json",
    "gradient_boosted_trees.json",
]
missing = [name for name in expected if not (result_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing result files: " + ", ".join(missing) + ". Run the three model notebooks first."
    )

frames = [json.loads((result_dir / name).read_text(encoding="utf-8")) for name in expected]
results = pd.DataFrame(frames)
columns = [
    "model", "best_cv_auc", "test_auc", "train_auc",
    "test_precision", "test_recall", "test_f1", "test_accuracy",
    "baseline_accuracy", "baseline_recall", "best_params",
]
display(results[columns])

## Chart

The cross-validated AUC is how each notebook chose its settings. The test bars are the held-out scores after that choice. A large gap between train AUC and test AUC means that notebook's model memorised the training customers.

In [ ]:
results = results.copy()
results["train_minus_test_auc"] = results["train_auc"] - results["test_auc"]
melted = results.melt(
    id_vars="model",
    value_vars=["best_cv_auc", "test_auc", "test_recall", "test_f1"],
    var_name="metric",
    value_name="score",
)
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=melted, x="metric", y="score", hue="model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Three models, same split. Settings were chosen by CV AUC.")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
display(results[["model", "train_auc", "test_auc", "train_minus_test_auc"]].round(3))

## How to talk about the merge

Each model notebook belongs to the person named at the top of that file, and that person should have committed it.
This comparison belongs to whoever ran the three results and checked that the split was the same seed.
Do not pick the winner from the test bars. The honest selection rule is the highest cross-validated AUC.
Then say whether that winner's train/test gap is small enough to trust.
The majority baseline predicts retained for everyone, so its churn recall is 0 and its accuracy is `baseline_accuracy`.